In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

In [ ]:
# Load the og dataset 
df = pd.read_csv("../data/raw/movies.csv")

print("Dataset shape:", df.shape)
df.head()

In [ ]:
# Make a copy of the OG dataset
movies = df.copy()

print("Original shape:", movies.shape)

In [ ]:
# Fix column names
print(movies.columns.tolist())

In [ ]:
movies = movies.rename(columns={
    "runtime,,": "runtime"
})

movies.columns.tolist()

In [ ]:
# Clean runtime 
movies["runtime"] = (
    movies["runtime"]
    .astype(str)
    .str.replace(",", "", regex=False)
)

movies["runtime"] = pd.to_numeric(
    movies["runtime"],
    errors="coerce"
)

print(movies["runtime"].head(20))
print("\nData type:", movies["runtime"].dtype)

In [ ]:
# Fix the budget problem 
movies["budget"] = movies["budget"].replace(0, np.nan) 
print("Missing budget:", movies["budget"].isna().sum())
print("Budget = 0:", (movies["budget"] == 0).sum())

In [ ]:
# Do NOT replace missing gross with zero 
movies["gross"] = movies["gross"].fillna(0) 

In [ ]:
# Cleaning the released column
movies["released_clean"] = (
    movies["released"]
    .astype("string")
    .str.replace(r"\s*\([^)]*\)", "", regex=True)
    .str.strip()
)
print(
    movies[["released", "released_clean"]]
    .head(20)
) 

In [ ]:
# Convert released date to datetime
movies["released_date"] = pd.to_datetime(
    movies["released_clean"],
    errors="coerce"
)
movies[["released_clean", "released_date"]].head(20)

In [ ]:
movies["released_date"].dtype

In [ ]:
# Check how many dates failed 
print(
    "Missing released dates:",
    movies["released_date"].isna().sum()
) 

In [ ]:
# Cleaning text columns 
text_columns = [
    "name",
    "rating",
    "genre",
    "director",
    "writer",
    "star",
    "country",
    "company"
]

for col in text_columns:
    movies[col] = movies[col].astype("string").str.strip()

In [ ]:
# converting appropriate text columns 
numeric_columns = [
    "year",
    "score",
    "votes",
    "budget",
    "gross",
    "runtime"
]

for col in numeric_columns:
    movies[col] = pd.to_numeric(
        movies[col],
        errors="coerce"
    )

In [ ]:
# check duplicates 
print("Duplicate rows:", movies.duplicated().sum())

In [ ]:
print(
    "Duplicate movie names:",
    movies["name"].duplicated().sum()
)

In [ ]:
# Creating a clean report 
cleaning_report = pd.DataFrame({
    "column": movies.columns,
    "dtype": movies.dtypes.astype(str).values,
    "missing_count": movies.isna().sum().values,
    "missing_percentage": (
        movies.isna().mean().values * 100
    ).round(2),
    "unique_values": movies.nunique().values
})

cleaning_report

In [ ]:
# Inspect the cleaned datasetabs
movies.head()

In [ ]:
movies.info()

In [ ]:
movies.info()

In [ ]:
print("Missing released dates:", movies["released_date"].isna().sum())

In [ ]:
print("Duplicate rows:", movies.duplicated().sum())

In [ ]:
print("Duplicate movie names:", movies["name"].duplicated().sum())

In [ ]:
print("Missing gross:", movies["gross"].isna().sum())
print("Gross = 0:", (movies["gross"] == 0).sum())

In [ ]:
print(
    movies[movies["gross"] == 0][
        ["name", "year", "budget", "gross", "score"]
    ].head(20)
)

In [ ]:
# Restore the originally missing gross values
movies["gross"] = movies["gross"].replace(0, np.nan)

In [ ]:
print("Missing gross:", movies["gross"].isna().sum())
print("Gross = 0:", (movies["gross"] == 0).sum())

In [ ]:
duplicate_names = movies[
    movies["name"].duplicated(keep=False)
].sort_values("name")

print("Number of duplicate-name rows:", len(duplicate_names))

duplicate_names[
    ["name", "year", "director", "genre", "gross"]
]

In [ ]:
missing_dates = movies[movies["released_date"].isna()]

print("Number of missing dates:", len(missing_dates))

missing_dates[
    ["name", "year", "released"]
].head(60)

In [ ]:
movies.iloc[-1]

In [ ]:
print(movies.iloc[-1].isna())

In [ ]:
print("Last row:")
print(movies.iloc[-1])

In [ ]:
print("\nMissing values in last row:")
print(movies.iloc[-1].isna())

In [ ]:
# Remove completely empty movie records
movies = movies.dropna(
    subset=[
        "name",
        "year",
        "released",
        "score",
        "votes",
        "director",
        "writer",
        "star",
        "country",
        "runtime"
    ],
    how="all"
)

print("Dataset shape:", movies.shape)

In [ ]:
movies = movies.drop(columns=["released_clean"])

In [ ]:
# Reset the index
movies = movies.reset_index(drop=True)

print(movies.shape)
print(movies.tail())

In [ ]:
# improve our release-date information
movies["release_year"] = movies["released_date"].dt.year
movies["release_month"] = movies["released_date"].dt.month

In [ ]:
# extract the year directly: 
movies["release_year"] = (
    movies["released"]
    .astype("string")
    .str.extract(r"(\d{4})")[0]
    .astype("Int64")
)


print(movies["release_year"].isna().sum())
print(movies[["year", "released", "release_year"]].head(20))

In [ ]:
# Extract release month 
month_pattern = (
    r"(January|February|March|April|May|June|July|August|"
    r"September|October|November|December)"
)

movies["release_month"] = (
    movies["released"]
    .astype("string")
    .str.extract(month_pattern, expand=False)
) 

print(
    movies[
        ["released", "release_year", "release_month"]
    ].head(20)
)

In [ ]:
# missing-value report 
final_quality_report = pd.DataFrame({
    "column": movies.columns,
    "dtype": movies.dtypes.astype(str).values,
    "missing_count": movies.isna().sum().values,
    "missing_percentage": (
        movies.isna().mean().values * 100
    ).round(2),
    "unique_values": movies.nunique().values
})

final_quality_report

In [ ]:
# Final duplicate check
print("Duplicate rows:", movies.duplicated().sum())
print("Duplicate movie names:", movies["name"].duplicated().sum())

In [ ]:
# Check the critical columns 
print("Dataset shape:", movies.shape)

print("\nBudget = 0:", (movies["budget"] == 0).sum())
print("Missing budget:", movies["budget"].isna().sum())

print("\nGross = 0:", (movies["gross"] == 0).sum())
print("Missing gross:", movies["gross"].isna().sum())

print("\nMissing runtime:", movies["runtime"].isna().sum())
print("Missing release date:", movies["released_date"].isna().sum())
print("Missing release year:", movies["release_year"].isna().sum())

In [ ]:
# Save the cleaned dataset 
output_path = "../data/processed/movies_cleaned.csv"

movies.to_csv(output_path, index=False)

print(f"Cleaned dataset saved to: {output_path}")